In [ ]:
!pip install evidently

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.0/238.0 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 568.3/568.3 kB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 49.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

In [ ]:
df = pd.read_csv("Churn_Preprocessed.csv")

print("✅ Preprocessed data loaded")

✅ Preprocessed data loaded


In [ ]:
# Reference (old data - training)
reference_data = df.sample(frac=0.7, random_state=42)

# Current (new incoming data)
current_data = df.sample(frac=0.3, random_state=1)

print("Reference shape:", reference_data.shape)
print("Current shape:", current_data.shape)

Reference shape: (7000, 12)
Current shape: (3000, 12)


In [ ]:
# Simulate drift (change distribution)
current_data["Age"] = current_data["Age"] + 10
current_data["Balance"] = current_data["Balance"] * 1.2

print("Artificial drift introduced")

Artificial drift introduced


In [ ]:
from evidently import Report
from evidently.presets import DataDriftPreset

report = Report(metrics=[DataDriftPreset()])

report.run(
    reference_data=reference_data,
    current_data=current_data
)

report

In [ ]:
# Simple statistical drift check (robust + always works)

drift_detected = False

for col in reference_data.columns:
    if col != "Exited":  # skip target
        ref_mean = reference_data[col].mean()
        curr_mean = current_data[col].mean()

        # If change > 20%, mark drift
        if abs(ref_mean - curr_mean) / (abs(ref_mean) + 1e-5) > 0.2:
            drift_detected = True
            print(f"Drift detected in feature: {col}")

print("\nFinal Drift Status:", drift_detected)

Drift detected in feature: Age
Drift detected in feature: Balance

Final Drift Status: True


In [ ]:
import joblib

if drift_detected:
    print("⚠️ Drift detected! Retraining model...")

    X = current_data.drop("Exited", axis=1)
    y = current_data["Exited"]

    from sklearn.ensemble import RandomForestClassifier
    model = RandomForestClassifier(random_state=42)
    model.fit(X, y)

    # 🔥 SAVE UPDATED MODEL (IMPORTANT)
    joblib.dump(model, "churn_model.pkl")

    print("✅ Model retrained and updated!")

⚠️ Drift detected! Retraining model...
✅ Model retrained and updated!
